In [ ]:
import logging
from datetime import datetime
from pyspark.sql import SparkDataFrame
from pyspark.sql.functions import col, count, when

logger = logging.getLogger("churn_model")


def validate_model(df: SparkDataFrame) -> SparkDataFrame:
    """Validate the model outputs."""
    logger.info(f"validate_model started at {datetime.now()}")
    
    # Validate input DataFrame is not empty
    if df.count() == 0:
        raise ValueError("Input DataFrame is empty")
    
    # Validate required columns exist
    required_columns = ["customer_id", "prediction", "probability"]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")
    
    # Validate prediction values are binary (0 or 1)
    invalid_predictions = df.filter(~col("prediction").isin([0, 1])).count()
    if invalid_predictions > 0:
        raise ValueError(f"Found {invalid_predictions} invalid prediction values (must be 0 or 1)")
    
    # Validate probability values are in [0, 1] range
    invalid_probs = df.filter((col("probability") < 0) | (col("probability") > 1)).count()
    if invalid_probs > 0:
        raise ValueError(f"Found {invalid_probs} probability values outside [0, 1] range")
    
    # Validate no null values in critical columns
    null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in required_columns]).collect()[0]
    for col_name in required_columns:
        if null_counts[col_name] > 0:
            raise ValueError(f"Found {null_counts[col_name]} null values in column '{col_name}'")
    
    logger.info(f"validate_model finished at {datetime.now()}")

    return df